# iBKH — Knowledge Graph Processing

All raw iBKH processed files are read from `BASE_PATH`.  
All reference/database files are read from `DB_BASE_PATH`.  
All output KG-triple CSVs are written to `OUT_PATH`.



## 0. Path Configuration

In [ ]:
import os

# ── Edit only these three lines ──────────────────────────────────────────────
BASE_PATH    = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"          # iBKH processed relation files
DB_BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases/"       # shared reference / database files
OUT_PATH     = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/iBKH/"           # all output KG CSVs land here
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_PATH, exist_ok=True)
print("BASE_PATH    :", BASE_PATH)
print("DB_BASE_PATH :", DB_BASE_PATH)
print("OUT_PATH     :", OUT_PATH)


## 1. Imports

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from glob import glob


## 2. Shared Helpers

In [129]:
def reorder(df, desired_cols):
    """Return df with only the desired columns that actually exist, in order."""
    existing = [c for c in desired_cols if c in df.columns]
    return df[existing]

KG_DESIRED_COLS = [
    "Head", "Relation", "Tail",
    "Head_type", "Tail_type",
    "KG_Source",
    "Head_detail_name", "Head_SMILES_name",
    "Tail_detail_name",
    "Head_ID_IS", "Tail_ID_IS",
]

def head_id_is(series):
    """Return 'Drugbank' if value starts with 'DB', else 'Pubchem', else None."""
    return np.where(
        series.isna(), None,
        np.where(series.astype(str).str.startswith('DB'), 'Drugbank', 'Pubchem')
    )

def strip_float_suffix(series):
    """Remove trailing .0 from numeric-looking strings."""
    return series.astype(str).str.replace(r'\.0$', '', regex=True)


## 3. Reference Dictionaries

### 3.1 PubChem CID → SMILES / IUPAC

In [ ]:
Pubchem = pd.read_pickle(os.path.join(DB_BASE_PATH, "PUBCHEM", "combined_df.pkl"))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_SMILES"]))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_IUPAC_NAME"]))
del Pubchem
print(f"Pubchem_CID_Smile_Dict : {len(Pubchem_CID_Smile_Dict):,} entries")
print(f"Pubchem_IUPAC_CID_Dict : {len(Pubchem_IUPAC_CID_Dict):,} entries")


### 3.2 DrugBank ↔ PubChem CID

In [ ]:
pubchem_db = pd.read_csv(os.path.join(DB_BASE_PATH, "PUBCHEM", "Pubchem_ID_DrugBank_Chebi.csv"))
pubchem_db_filt = pubchem_db[~pubchem_db["DRUGBANK_ID"].isna()]
DB2PC_dict  = dict(zip(pubchem_db_filt["DRUGBANK_ID"], pubchem_db_filt["ID"]))
DB2Pub_dict = dict(zip(pubchem_db_filt["ID"],          pubchem_db_filt["DRUGBANK_ID"]))

pubchem_chebi = pubchem_db[~pubchem_db["CHEBI_ID"].isna()]
Chebi2PC_dict = dict(zip(pubchem_chebi["CHEBI_ID"], pubchem_chebi["ID"]))
del pubchem_db, pubchem_db_filt, pubchem_chebi
print(f"DB2PC_dict   : {len(DB2PC_dict):,} entries")
print(f"Chebi2PC_dict: {len(Chebi2PC_dict):,} entries")


In [127]:
drugbank = pd.read_csv(os.path.join(DB_BASE_PATH, "drugbank", "drugbank_all.csv"))
drugbank_dict = dict(zip(drugbank['drugbank_id'],drugbank['name']))
# drugbank_dict

### 3.3 Reactome Pathways

In [ ]:
pathway_reactome = pd.read_csv(
    os.path.join(DB_BASE_PATH, "Reactome", "ReactomePathways.txt"),
    sep="\t", header=None
)
pathway_reactome = pathway_reactome[pathway_reactome[0].str.startswith("R-HSA")]
pathway_reactome_dict     = dict(zip(pathway_reactome[0], pathway_reactome[1]))
pathway_reactome_rev_dict = dict(zip(pathway_reactome[1], pathway_reactome[0]))
print(f"Reactome pathways: {len(pathway_reactome_dict):,}")


### 3.4 NCBI Gene dictionaries

In [ ]:
ENS_2NCBI = pd.read_csv(os.path.join(DB_BASE_PATH, "ENSEMBL", "ENSEMBLE_ID_2_NCBI_Gene_SYM.csv"))
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI["name"].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI["name"], ENS_2NCBI["initial_alias"]))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(
    os.path.join(DB_BASE_PATH, "NCBI", "Homo_sapiens.gene_info"), sep="\t"
)

def extract_hgnc(value):
    match = re.search(r'\|?(HGNC:HGNC:\d+)\|?', value)
    return match.group(1) if match else None

NCBI_ALL_GENE["HGNC"]       = NCBI_ALL_GENE["dbXrefs"].apply(
    lambda x: extract_hgnc(x) if isinstance(x, str) else None
)
NCBI_ALL_GENE["ENSEMBLE_ID"] = NCBI_ALL_GENE["Symbol"].map(NCBI_2ENS__dict)

NCBI_ALL_GENEname_dict     = dict(zip(NCBI_ALL_GENE["GeneID"],      NCBI_ALL_GENE["description"]))
NCBI_ALL_GENEIDname_dict   = dict(zip(NCBI_ALL_GENE["GeneID"],      NCBI_ALL_GENE["Symbol"]))
NCBI_Synbol_GENEname_dict  = dict(zip(NCBI_ALL_GENE["Symbol"],      NCBI_ALL_GENE["description"]))

HGNC_NCBI = NCBI_ALL_GENE[~NCBI_ALL_GENE["HGNC"].isna()].copy()
HGNC_NCBI["HGNC"] = HGNC_NCBI["HGNC"].str.replace("^HGNC:", "", regex=True)
HGNC_NCBI_dict = dict(zip(HGNC_NCBI["HGNC"], HGNC_NCBI["Symbol"]))
del NCBI_ALL_GENE, HGNC_NCBI
print(f"NCBI_Synbol_GENEname_dict : {len(NCBI_Synbol_GENEname_dict):,}")
print(f"HGNC_NCBI_dict            : {len(HGNC_NCBI_dict):,}")


### 3.5 Disease Ontology (DO / DOID)

In [ ]:
DO_All_File = pd.read_csv(os.path.join(DB_BASE_PATH, "DO", "All_DO_ref_files.csv"))
DOID_disname_dict      = dict(zip(DO_All_File["id"],    DO_All_File["label"]))
DOID_disAltID_dict     = dict(zip(DO_All_File["id"],    DO_All_File["xrefs"]))
DOID_disname_DOID_dict = dict(zip(DO_All_File["label"], DO_All_File["id"]))
print(f"DOID_disname_dict : {len(DOID_disname_dict):,}")


### 3.6 OMIM → DOID

In [ ]:
omim_raw = pd.read_csv(
    os.path.join(DB_BASE_PATH, "DO", "HumanDiseaseOntology-main", "DOreports", "OMIMinDO.tsv"),
    sep="\t"
)
omim_raw.rename(columns={"ID": "id", "LABEL": "label"}, inplace=True)
omim_raw["xrefs"] = omim_raw["xrefs"].str.split(", ")
omim_raw = omim_raw.explode("xrefs")
omim_raw["xrefs"] = omim_raw["xrefs"].str.replace("MIM", "OMIM")
OMIM_ID_2_DOID_dict = dict(zip(omim_raw["xrefs"], omim_raw["id"]))
del omim_raw
print(f"OMIM_ID_2_DOID_dict : {len(OMIM_ID_2_DOID_dict):,}")


### 3.7 MeSH → DOID

In [ ]:
mesh2doid_raw = pd.read_csv(
    os.path.join(DB_BASE_PATH, "DO", "HumanDiseaseOntology-main", "DOreports", "MESHinDO.tsv"),
    sep="\t"
)
mesh2doid_raw.rename(columns={"ID": "id", "LABEL": "label"}, inplace=True)
mesh2doid_raw["xrefs"] = mesh2doid_raw["xrefs"].str.split(", ")
mesh2doid_raw = mesh2doid_raw.explode("xrefs")
Mesh2DOID_dict   = dict(zip(mesh2doid_raw["xrefs"], mesh2doid_raw["id"]))
MESH2Name_Dict_1 = dict(zip(mesh2doid_raw["xrefs"], mesh2doid_raw["label"]))
del mesh2doid_raw
print(f"Mesh2DOID_dict : {len(Mesh2DOID_dict):,}")


### 3.8 MeSH Supplementary (ID → Name / PubChem)

In [ ]:
Mesh_supp = pd.read_csv(os.path.join(DB_BASE_PATH, "MESH", "Mesh_Supplementary_Info.csv"))
Mesh_supp_dict = dict(zip(Mesh_supp["ID"], Mesh_supp["Name"]))

Mesh_pubchem_supp = Mesh_supp[~Mesh_supp["Pubchem_ID"].isna()].copy()
Mesh_pubchem_supp["Pubchem_ID"] = strip_float_suffix(Mesh_pubchem_supp["Pubchem_ID"])
MESH_pubchem_Supp_dict = dict(zip(Mesh_pubchem_supp["ID"], Mesh_pubchem_supp["Pubchem_ID"]))
del Mesh_supp, Mesh_pubchem_supp
print(f"Mesh_supp_dict         : {len(Mesh_supp_dict):,}")
print(f"MESH_pubchem_Supp_dict : {len(MESH_pubchem_Supp_dict):,}")


### 3.9 HPO Phenotype / Side-Effect

In [ ]:
phenotype = pd.read_csv(os.path.join(DB_BASE_PATH, "HPO", "HPO.csv"))
phenotype = phenotype[phenotype["id"].str.startswith("HP")]
phenotype_dict       = dict(zip(phenotype["name"], phenotype["id"]))
HPOphenotype_name_dict = dict(zip(phenotype["id"],   phenotype["name"]))
del phenotype
print(f"HPOphenotype_name_dict : {len(HPOphenotype_name_dict):,}")


### 3.10 MedGen CUI → Source ID

In [ ]:
MedGen_raw = pd.read_csv(
    os.path.join(DB_BASE_PATH, "MESH", "MeSH", "MedGen_HPO_Mapping.txt"), sep="|"
)
MedGen_CUID_Source_ID_dict = dict(zip(MedGen_raw["#CUI"], MedGen_raw["SDUI"]))
del MedGen_raw
print(f"MedGen_CUID_Source_ID_dict : {len(MedGen_CUID_Source_ID_dict):,}")


In [ ]:
MedGen_raw = pd.read_csv(
    os.path.join(BASE_PATH, "MESH", "MeSH", "MedGen_HPO_Mapping.txt"), sep="|"
)

In [144]:
gene_vocab = pd.read_csv(
    os.path.join(BASE_PATH, "ibkh", "vocabulary_data", "gene_vocab.csv"), sep=","
)
gene_vocab_dict = dict(zip(gene_vocab["primary"], gene_vocab["symbol"]))
del gene_vocab
print(f"MedGen_CUID_Source_ID_dict : {len(gene_vocab_dict):,}")
gene_vocab_dict

MedGen_CUID_Source_ID_dict : 88,211


{'HGNC:5': 'A1BG',
 'HGNC:37133': 'A1BG-AS1',
 'HGNC:24086': 'A1CF',
 'HGNC:7': 'A2M',
 'HGNC:27057': 'A2M-AS1',
 'HGNC:23336': 'A2ML1',
 'HGNC:41022': 'A2ML1-AS1',
 'HGNC:41523': 'A2ML1-AS2',
 'HGNC:8': 'A2MP1',
 'HGNC:30005': 'A3GALT2',
 'HGNC:18149': 'A4GALT',
 'HGNC:17968': 'A4GNT',
 'HGNC:1': 'A12M1',
 'HGNC:2': 'A12M2',
 'HGNC:3': 'A12M3',
 'HGNC:4': 'A12M4',
 'HGNC:13666': 'AAAS',
 'HGNC:12': 'AABT',
 'HGNC:21298': 'AACS',
 'HGNC:18226': 'AACSP1',
 'HGNC:17': 'AADAC',
 'HGNC:24427': 'AADACL2',
 'HGNC:50301': 'AADACL2-AS1',
 'HGNC:32037': 'AADACL3',
 'HGNC:32038': 'AADACL4',
 'HGNC:50305': 'AADACP1',
 'HGNC:17929': 'AADAT',
 'HGNC:25662': 'AAGAB',
 'HGNC:19679': 'AAK1',
 'HGNC:30205': 'AAMDC',
 'HGNC:18': 'AAMP',
 'HGNC:19': 'AANAT',
 'HGNC:15886': 'AAR2',
 'HGNC:33842': 'AARD',
 'HGNC:20': 'AARS1',
 'HGNC:49894': 'AARS1P1',
 'HGNC:21022': 'AARS2',
 'HGNC:28417': 'AARSD1',
 'HGNC:54994': 'AARSD1P1',
 'HGNC:23993': 'AASDH',
 'HGNC:14235': 'AASDHPPT',
 'HGNC:17366': 'AASS',
 'HGNC:

In [262]:
kegg_pathway =  pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/kegg/kegg_pathways.txt',
    sep="\t",
    header=None,
    names=["KEGG_ID", "Pathway_Name"]
)
kegg_pathway["KEGG_ID"] = "KEGG:" + kegg_pathway["KEGG_ID"]
kegg_pathway_dict = dict(zip(kegg_pathway['KEGG_ID'],kegg_pathway['Pathway_Name']))
kegg_pathway_dict
# kegg_pathway

{'KEGG:map01100': 'Metabolic pathways',
 'KEGG:map01110': 'Biosynthesis of secondary metabolites',
 'KEGG:map01120': 'Microbial metabolism in diverse environments',
 'KEGG:map01200': 'Carbon metabolism',
 'KEGG:map01210': '2-Oxocarboxylic acid metabolism',
 'KEGG:map01212': 'Fatty acid metabolism',
 'KEGG:map01230': 'Biosynthesis of amino acids',
 'KEGG:map01232': 'Nucleotide metabolism',
 'KEGG:map01250': 'Biosynthesis of nucleotide sugars',
 'KEGG:map01240': 'Biosynthesis of cofactors',
 'KEGG:map01220': 'Degradation of aromatic compounds',
 'KEGG:map01310': 'Nitrogen cycle',
 'KEGG:map01320': 'Sulfur cycle',
 'KEGG:map00010': 'Glycolysis / Gluconeogenesis',
 'KEGG:map00020': 'Citrate cycle (TCA cycle)',
 'KEGG:map00030': 'Pentose phosphate pathway',
 'KEGG:map00040': 'Pentose and glucuronate interconversions',
 'KEGG:map00051': 'Fructose and mannose metabolism',
 'KEGG:map00052': 'Galactose metabolism',
 'KEGG:map00053': 'Ascorbate and aldarate metabolism',
 'KEGG:map00500': 'Starch

In [220]:
bto_tissue =  pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/bto/BTO_ALL_COMBINED.csv')#,
bto_tissue

bto_tissue_dict = dict(zip(bto_tissue['ID'],bto_tissue['name']))
bto_tissue_dict

{'BTO:0000000': 'tissues, cell types and enzyme sources',
 'BTO:0000001': 'culture condition:-induced cell',
 'BTO:0000002': 'culture condition:1,4-dichlorobenzene-grown cell',
 'BTO:0000003': 'intestinal cell line',
 'BTO:0000004': 'culture condition:2,5-dihydroxybenzoate-grown cell',
 'BTO:0000005': 'culture condition:2-aminobenzenesulfonate-grown cell',
 'BTO:0000006': 'osteoblastoma cell',
 'BTO:0000007': 'HEK-293 cell',
 'BTO:0000008': 'culture condition:3-chlorobenzoate-grown cell',
 'BTO:0000009': 'culture condition:3-hydroxybenzoate-grown cell',
 'BTO:0000010': 'culture condition:3-methylcrotonoyl-CoA-grown cell',
 'BTO:0000011': '3T3-L1 cell',
 'BTO:0000012': 'culture condition:4-(methoxymethyl)phenol-grown cell',
 'BTO:0000013': 'culture condition:4-chlorophenol-grown cell',
 'BTO:0000014': 'culture condition:4-hydroxybenzoate-grown cell',
 'BTO:0000015': 'culture condition:4-methylmuconolactone-grown cell',
 'BTO:0000016': 'A-172 cell',
 'BTO:0000017': 'A-431 cell',
 'BTO:00

In [225]:
uberon =  pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv')#,
uberon = uberon[~uberon['Name'].isna()]

uberon_dict = dict(zip(uberon['UBERON ID'],uberon['Name']))
uberon_dict

{'UBERON:0005200': 'thoracic mammary gland',
 'UBERON:0005278': 'nail bed of finger',
 'UBERON:0010953': 'nasalis muscle',
 'UBERON:0007224': 'medial entorhinal cortex',
 'UBERON:0001069': 'head of pancreas',
 'UBERON:0015215': 'median arcuate ligament',
 'UBERON:3000998': 'suprarostral cartilage',
 'UBERON:0006849': 'scapula',
 'UBERON:0000937': 'obsolete proneural cluster',
 'UBERON:0012239': 'urinary bladder vasculature',
 'UBERON:0001883': 'olfactory tubercle',
 'UBERON:0005250': 'stomatodeum gland',
 'UBERON:0004745': 'parasphenoid',
 'UBERON:0012262': 'manual minor digit (Aves)',
 'UBERON:0034735': 'oviduct albumen gland',
 'UBERON:0003143': 'pupa',
 'UBERON:0018146': 'transverse process of lumbar vertebra',
 'UBERON:0014667': 'periventricular nucleus of hypothalamus',
 'UBERON:0003011': 'facial motor nucleus',
 'UBERON:0011685': 'preorbitalis muscle',
 'UBERON:0004177': 'hemopoietic organ',
 'UBERON:0009957': 'ciliated pit',
 'UBERON:0002101': 'limb',
 'UBERON:0003667': 'lower j

### 3.11 Shared `map_drug_head_id` helper


In [ ]:
def map_drug_head_id(value):
    """Map Head compound ID (DrugBank DB* or MeSH) to PubChem CID."""
    if not isinstance(value, str):
        return None
    if value.startswith("DB"):
        return DB2PC_dict.get(value)
    # MeSH → PubChem via supplementary dict
    return MESH_pubchem_Supp_dict.get(value)


In [121]:
kegg_mapping = pd.read_csv(
    os.path.join(DB_BASE_PATH, "kegg", "kegg_pathways.txt"),
    sep="\t",
    header=None,
    names=["KEGG_ID", "Pathway_Name"]
)

kegg_mapping_dict = dict(zip(kegg_mapping["KEGG_ID"], kegg_mapping["Pathway_Name"]))
kegg_mapping_dict

{'map01100': 'Metabolic pathways',
 'map01110': 'Biosynthesis of secondary metabolites',
 'map01120': 'Microbial metabolism in diverse environments',
 'map01200': 'Carbon metabolism',
 'map01210': '2-Oxocarboxylic acid metabolism',
 'map01212': 'Fatty acid metabolism',
 'map01230': 'Biosynthesis of amino acids',
 'map01232': 'Nucleotide metabolism',
 'map01250': 'Biosynthesis of nucleotide sugars',
 'map01240': 'Biosynthesis of cofactors',
 'map01220': 'Degradation of aromatic compounds',
 'map01310': 'Nitrogen cycle',
 'map01320': 'Sulfur cycle',
 'map00010': 'Glycolysis / Gluconeogenesis',
 'map00020': 'Citrate cycle (TCA cycle)',
 'map00030': 'Pentose phosphate pathway',
 'map00040': 'Pentose and glucuronate interconversions',
 'map00051': 'Fructose and mannose metabolism',
 'map00052': 'Galactose metabolism',
 'map00053': 'Ascorbate and aldarate metabolism',
 'map00500': 'Starch and sucrose metabolism',
 'map00620': 'Pyruvate metabolism',
 'map00630': 'Glyoxylate and dicarboxylate 

## 4. Drug–Drug Interactions


In [134]:
ibkh_D_D = pd.read_csv(os.path.join(BASE_PATH, "ibkh/D_D_res.csv"))
ibkh_D_D = ibkh_D_D.rename(columns={
    "Drug_1": "Head",
    "Drug_2": "Tail"
})

ibkh_D_D["Head"] = ibkh_D_D["Head"].str.replace("^DrugBank:", "", regex=True)
ibkh_D_D["Tail"] = ibkh_D_D["Tail"].str.replace("^DrugBank:", "", regex=True)

ibkh_D_D["Head_ID"] = ibkh_D_D["Head"].map(DB2PC_dict).fillna(ibkh_D_D["Head"])
ibkh_D_D["Tail_ID"] = ibkh_D_D["Tail"].map(DB2PC_dict).fillna(ibkh_D_D["Tail"])
ibkh_D_D["Head_ID"] = ibkh_D_D["Head_ID"].astype(str).str.replace(r'\.0$', '', regex=True)
ibkh_D_D["Tail_ID"] = ibkh_D_D["Tail_ID"].astype(str).str.replace(r'\.0$', '', regex=True)

ibkh_D_D[["Head", "Head_ID"]] = ibkh_D_D[["Head_ID", "Head"]]
ibkh_D_D[["Tail", "Tail_ID"]] = ibkh_D_D[["Tail_ID", "Tail"]]

ibkh_D_D["Head_detail_name"] = ibkh_D_D["Head"].map(Pubchem_IUPAC_CID_Dict).fillna(ibkh_D_D["Head"].map(drugbank_dict))
ibkh_D_D["Tail_detail_name"] = ibkh_D_D["Tail"].map(Pubchem_IUPAC_CID_Dict).fillna(ibkh_D_D["Tail"].map(drugbank_dict))
ibkh_D_D["Head_SMILES_name"] = ibkh_D_D["Head"].map(Pubchem_CID_Smile_Dict)
ibkh_D_D["Tail_SMILES_name"] = ibkh_D_D["Tail"].map(Pubchem_CID_Smile_Dict)

ibkh_D_D["Head_ID_IS"] = head_id_is(ibkh_D_D["Head"])
ibkh_D_D["Tail_ID_IS"] = head_id_is(ibkh_D_D["Tail"])
ibkh_D_D["Head_type"]  = "ChemicalEntity"
ibkh_D_D["Tail_type"]  = "ChemicalEntity"
ibkh_D_D["Relation"]   = ibkh_D_D["Head_type"] + "_" + ibkh_D_D["Tail_type"]
ibkh_D_D["KG_Source"]  = "iBKH"

ibkh_D_D = reorder(ibkh_D_D, KG_DESIRED_COLS)
print(f"Drug–Drug rows: {len(ibkh_D_D):,}")
ibkh_D_D

Drug–Drug rows: 2,684,682


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILES_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,DB00001,ChemicalEntity_ChemicalEntity,10182969,ChemicalEntity,ChemicalEntity,iBKH,Lepirudin,NaN,1-(4-methoxyphenyl)-7-oxo-6-[4-(2-oxopiperidin...,Drugbank,Pubchem
1,DB00001,ChemicalEntity_ChemicalEntity,135565674,ChemicalEntity,ChemicalEntity,iBKH,Lepirudin,NaN,ethyl 3-[[2-[[4-(N-hexoxycarbonylcarbamimidoyl...,Drugbank,Pubchem
2,DB00001,ChemicalEntity_ChemicalEntity,3062316,ChemicalEntity,ChemicalEntity,iBKH,Lepirudin,NaN,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,Drugbank,Pubchem
3,DB00001,ChemicalEntity_ChemicalEntity,214348,ChemicalEntity,ChemicalEntity,iBKH,Lepirudin,NaN,"4-[3,5-bis(2-hydroxyphenyl)-1,2,4-triazol-1-yl...",Drugbank,Pubchem
4,DB00001,ChemicalEntity_ChemicalEntity,31401,ChemicalEntity,ChemicalEntity,iBKH,Lepirudin,NaN,"(4R)-4-[(3R,5S,7S,8R,9S,10S,13R,14S,17R)-3,7-d...",Drugbank,Pubchem
...,...,...,...,...,...,...,...,...,...,...,...
2684677,34359,ChemicalEntity_ChemicalEntity,3779,ChemicalEntity,ChemicalEntity,iBKH,"(2S)-3-(3,4-dihydroxyphenyl)-2-hydrazinyl-2-me...",C[C@](CC1=CC(=C(C=C1)O)O)(C(=O)O)NN,4-[1-hydroxy-2-(propan-2-ylamino)ethyl]benzene...,Pubchem,Pubchem
2684678,51081,ChemicalEntity_ChemicalEntity,10178705,ChemicalEntity,ChemicalEntity,iBKH,1-ethyl-6-fluoro-7-(4-methylpiperazin-1-yl)-4-...,CCN1C=C(C(=O)C2=CC(=C(C=C21)N3CCN(CC3)C)F)C(=O)O,7-[(3R)-3-aminoazepan-1-yl]-8-chloro-1-cyclopr...,Pubchem,Pubchem
2684679,135403821,ChemicalEntity_ChemicalEntity,6436173,ChemicalEntity,ChemicalEntity,iBKH,"[(7S,9E,11S,12R,13S,14R,15R,16R,17S,18S,19E,21...",C[C@H]1/C=C/C=C(\C(=O)NC2=C(C(=C3C(=C2O)C(=C(C...,"[(7S,9E,11S,12R,13S,14R,15R,16R,17S,18S,19E,21...",Pubchem,Pubchem
2684680,5340,ChemicalEntity_ChemicalEntity,9047,ChemicalEntity,ChemicalEntity,iBKH,"4-amino-N-(1,3-thiazol-2-yl)benzenesulfonamide",C1=CC(=CC=C1N)S(=O)(=O)NC2=NC=CS2,4-amino-N-(3-methoxypyrazin-2-yl)benzenesulfon...,Pubchem,Pubchem


In [135]:
ibkh_D_D.to_csv(os.path.join(OUT_PATH, "Chemical_Chemical_ibkh.csv"), index=False)
print("Saved: Chemical_Chemical_ibkh.csv")
# del ibkh_D_D

Saved: Chemical_Chemical_ibkh.csv


## 5. Drug–Gene Interactions


In [146]:
ibkh_D_G = pd.read_csv(os.path.join(BASE_PATH, "ibkh/D_G_res.csv"))
 
ibkh_D_G  = ibkh_D_G.rename(columns={
    "Drug": "Head",
    "Gene": "Tail"
})
ibkh_D_G["Head"] = ibkh_D_G["Head"].str.replace("^DrugBank:", "", regex=True)

ibkh_D_G["Head_ID"] = ibkh_D_G["Head"].map(DB2PC_dict).fillna(ibkh_D_G["Head"])
ibkh_D_G["Head_ID"] = strip_float_suffix(ibkh_D_G["Head_ID"])  

ibkh_D_G[["Head", "Head_ID"]] = ibkh_D_G[["Head_ID", "Head"]]
ibkh_D_G["Head_detail_name"] = ibkh_D_G["Head"].map(Pubchem_IUPAC_CID_Dict).fillna(ibkh_D_G["Head"])
ibkh_D_G["Head_SMILES_name"] = ibkh_D_G["Head"].map(Pubchem_CID_Smile_Dict)

ibkh_D_G

,Head,Tail,Target,Transporter,Enzyme,Carrier,Downregulates,Upregulates,Associate,Binds,...,increases expression/production,"binding, ligand (esp. receptors)",decreases expression/production,"transport, channels",enzyme activity,physical association,Source,Head_ID,Head_detail_name,Head_SMILES_name
0,1051,HGNC:4855,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,DrugBank,DB00114,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,CC1=NC=C(C(=C1O)C=O)COP(=O)(O)O
1,6274,HGNC:4855,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,DRKG;DrugBank;Hetinoet,DB00117,(2S)-2-amino-3-(1H-imidazol-5-yl)propanoic acid,C1=C(NC=N1)C[C@@H](C(=O)O)N
2,33032,HGNC:29570,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,DrugBank,DB00142,(2S)-2-aminopentanedioic acid,C(CC(=O)O)[C@@H](C(=O)O)N
3,65249,HGNC:3531,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,DrugBank,DB02340,(2S)-2-acetamido-3-hydroxypropanoic acid,CC(=O)N[C@@H](CO)C(=O)O
4,DB11300,HGNC:3531,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,DrugBank,DB11300,DB11300,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1303742,5291,HGNC:29650,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...
1303743,5291,HGNC:18591,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...
1303744,5291,HGNC:11088,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...
1303745,5291,HGNC:16870,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...


In [147]:
ibkh_D_G_save = ibkh_D_G

In [148]:
ibkh_D_G["Tail"] = ibkh_D_G["Tail"].str.replace("^NCBI:", "", regex=True)
ibkh_D_G["Tail_ID"] = (
    ibkh_D_G["Tail"].map(HGNC_NCBI_dict)
    .fillna(ibkh_D_G["Tail"].map(NCBI_ALL_GENEIDname_dict))
)
ibkh_D_G

,Head,Tail,Target,Transporter,Enzyme,Carrier,Downregulates,Upregulates,Associate,Binds,...,"binding, ligand (esp. receptors)",decreases expression/production,"transport, channels",enzyme activity,physical association,Source,Head_ID,Head_detail_name,Head_SMILES_name,Tail_ID
0,1051,HGNC:4855,1,0,0,0,0,0,0,0,...,0,0,0,0,0,DrugBank,DB00114,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,CC1=NC=C(C(=C1O)C=O)COP(=O)(O)O,HDC
1,6274,HGNC:4855,1,0,0,0,0,0,0,1,...,0,0,0,0,0,DRKG;DrugBank;Hetinoet,DB00117,(2S)-2-amino-3-(1H-imidazol-5-yl)propanoic acid,C1=C(NC=N1)C[C@@H](C(=O)O)N,HDC
2,33032,HGNC:29570,1,0,0,0,0,0,0,0,...,0,0,0,0,0,DrugBank,DB00142,(2S)-2-aminopentanedioic acid,C(CC(=O)O)[C@@H](C(=O)O)N,GLS2
3,65249,HGNC:3531,1,0,0,0,0,0,0,0,...,0,0,0,0,0,DrugBank,DB02340,(2S)-2-acetamido-3-hydroxypropanoic acid,CC(=O)N[C@@H](CO)C(=O)O,F13A1
4,DB11300,HGNC:3531,1,0,0,0,0,0,0,0,...,0,0,0,0,0,DrugBank,DB11300,DB11300,NaN,F13A1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1303742,5291,HGNC:29650,0,0,0,0,0,0,1,0,...,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,PKMYT1
1303743,5291,HGNC:18591,0,0,0,0,0,0,1,0,...,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,NEK9
1303744,5291,HGNC:11088,0,0,0,0,0,0,1,0,...,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,SLK
1303745,5291,HGNC:16870,0,0,0,0,0,0,1,0,...,0,0,0,0,0,DRKG,DB00619,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,MELK


In [149]:
# ibkh_D_G["Tail_ID"] = ibkh_D_G["Tail"].map(gene_vocab_dict).fillna(ibkh_D_G["Tail"])
ibkh_D_G[["Tail", "Tail_ID"]] = ibkh_D_G[["Tail_ID", "Tail"]]
ibkh_D_G = ibkh_D_G[ibkh_D_G["Tail"].notna()]
ibkh_D_G["Tail_detail_name"] = ibkh_D_G["Tail"].map(NCBI_Synbol_GENEname_dict)

ibkh_D_G["Head_ID_IS"] = head_id_is(ibkh_D_G["Head"])
ibkh_D_G["Tail_ID_IS"] = "NCBI_ID"
ibkh_D_G["Head_type"]  = "ChemicalEntity"
ibkh_D_G["Tail_type"]  = "Gene"
ibkh_D_G["Relation"]   = ibkh_D_G["Head_type"] + "_" + ibkh_D_G["Tail_type"]
ibkh_D_G["KG_Source"]  = "iBKH"

ibkh_D_G = reorder(ibkh_D_G, KG_DESIRED_COLS)
print(f"Drug–Gene rows: {len(ibkh_D_G):,}")
ibkh_D_G

/tmp/ipykernel_1690022/389741019.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ibkh_D_G["Tail_detail_name"] = ibkh_D_G["Tail"].map(NCBI_Synbol_GENEname_dict)
/tmp/ipykernel_1690022/389741019.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ibkh_D_G["Head_ID_IS"] = head_id_is(ibkh_D_G["Head"])
/tmp/ipykernel_1690022/389741019.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

Drug–Gene rows: 1,172,214


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILES_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,1051,ChemicalEntity_Gene,HDC,ChemicalEntity,Gene,iBKH,(4-formyl-5-hydroxy-6-methylpyridin-3-yl)methy...,CC1=NC=C(C(=C1O)C=O)COP(=O)(O)O,histidine decarboxylase,Pubchem,NCBI_ID
1,6274,ChemicalEntity_Gene,HDC,ChemicalEntity,Gene,iBKH,(2S)-2-amino-3-(1H-imidazol-5-yl)propanoic acid,C1=C(NC=N1)C[C@@H](C(=O)O)N,histidine decarboxylase,Pubchem,NCBI_ID
2,33032,ChemicalEntity_Gene,GLS2,ChemicalEntity,Gene,iBKH,(2S)-2-aminopentanedioic acid,C(CC(=O)O)[C@@H](C(=O)O)N,glutaminase 2,Pubchem,NCBI_ID
3,65249,ChemicalEntity_Gene,F13A1,ChemicalEntity,Gene,iBKH,(2S)-2-acetamido-3-hydroxypropanoic acid,CC(=O)N[C@@H](CO)C(=O)O,coagulation factor XIII A chain,Pubchem,NCBI_ID
4,DB11300,ChemicalEntity_Gene,F13A1,ChemicalEntity,Gene,iBKH,DB11300,NaN,coagulation factor XIII A chain,Drugbank,NCBI_ID
...,...,...,...,...,...,...,...,...,...,...,...
1303742,5291,ChemicalEntity_Gene,PKMYT1,ChemicalEntity,Gene,iBKH,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,"protein kinase, membrane associated tyrosine/t...",Pubchem,NCBI_ID
1303743,5291,ChemicalEntity_Gene,NEK9,ChemicalEntity,Gene,iBKH,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,NIMA related kinase 9,Pubchem,NCBI_ID
1303744,5291,ChemicalEntity_Gene,SLK,ChemicalEntity,Gene,iBKH,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,STE20 like kinase,Pubchem,NCBI_ID
1303745,5291,ChemicalEntity_Gene,MELK,ChemicalEntity,Gene,iBKH,4-[(4-methylpiperazin-1-yl)methyl]-N-[4-methyl...,CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C...,maternal embryonic leucine zipper kinase,Pubchem,NCBI_ID


In [150]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/iBKH/'

In [151]:
ibkh_D_G.to_csv(os.path.join(OUT_PATH, "Chemical_Gene_ibkh.csv"), index=False)
print("Saved: Chemical_Gene_ibkh.csv")
del ibkh_D_G

Saved: Chemical_Gene_ibkh.csv


## 6. Drug–Disease Interactions

In [155]:
ibkh_D_Disease = pd.read_csv(os.path.join(BASE_PATH, "ibkh/D_Di_res.csv"))
ibkh_D_Disease  = ibkh_D_Disease.rename(columns={
    "Drug": "Head",
    "Disease": "Tail"
})
prefixes_drug = ("MeSH", "Drug", "DB08", "DB01")
ibkh_D_Disease = ibkh_D_Disease[ibkh_D_Disease["Head"].str.startswith(prefixes_drug, na=False)]
ibkh_D_Disease["Head"] = ibkh_D_Disease["Head"].str.replace("^DrugBank:", "", regex=True)
ibkh_D_Disease = ibkh_D_Disease[ibkh_D_Disease["Tail"].str.startswith("DOID")]

ibkh_D_Disease["Head_ID"] = ibkh_D_Disease["Head"].apply(map_drug_head_id)  
ibkh_D_Disease["Head_ID"] = strip_float_suffix(ibkh_D_Disease["Head_ID"])   
ibkh_D_Disease = ibkh_D_Disease[ibkh_D_Disease["Head_ID"].notna() & (ibkh_D_Disease["Head_ID"] != "nan")]

ibkh_D_Disease[["Head", "Head_ID"]] = ibkh_D_Disease[["Head_ID", "Head"]]
ibkh_D_Disease["Head_detail_name"] = ibkh_D_Disease["Head"].map(Pubchem_IUPAC_CID_Dict).fillna(ibkh_D_Disease["Head"])
ibkh_D_Disease["Head_SMILES_name"] = ibkh_D_Disease["Head"].map(Pubchem_CID_Smile_Dict)

ibkh_D_Disease["Tail_detail_name"] = ibkh_D_Disease["Tail"].map(DOID_disname_dict)
ibkh_D_Disease = ibkh_D_Disease[ibkh_D_Disease["Tail_detail_name"].notna()]

ibkh_D_Disease["Head_ID_IS"] = head_id_is(ibkh_D_Disease["Head"])
ibkh_D_Disease["Tail_ID_IS"] = "DOID"
ibkh_D_Disease["Head_type"]  = "ChemicalEntity"
ibkh_D_Disease["Tail_type"]  = "Disease"
ibkh_D_Disease["Relation"]   = ibkh_D_Disease["Head_type"] + "_" + ibkh_D_Disease["Tail_type"]
ibkh_D_Disease["KG_Source"]  = "iBKH"

ibkh_D_Disease = reorder(ibkh_D_Disease, KG_DESIRED_COLS)
print(f"Drug–Disease rows: {len(ibkh_D_Disease):,}")
ibkh_D_Disease
ibkh_D_Disease

Drug–Disease rows: 545,470


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILES_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,31703,ChemicalEntity_Disease,DOID:363,ChemicalEntity,Disease,iBKH,"(7S,9S)-7-[(2R,4S,5S,6S)-4-amino-5-hydroxy-6-m...",C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...,uterine cancer,Pubchem,DOID
1,5770,ChemicalEntity_Disease,DOID:10763,ChemicalEntity,Disease,iBKH,"methyl (1R,15S,17R,18R,19S,20S)-6,18-dimethoxy...",CO[C@H]1[C@@H](C[C@@H]2CN3CCC4=C([C@H]3C[C@@H]...,hypertension,Pubchem,DOID
2,4828,ChemicalEntity_Disease,DOID:10763,ChemicalEntity,Disease,iBKH,1-(1H-indol-4-yloxy)-3-(propan-2-ylamino)propa...,CC(C)NCC(COC1=CC=CC2=C1C=CN2)O,hypertension,Pubchem,DOID
3,4493,ChemicalEntity_Disease,DOID:10283,ChemicalEntity,Disease,iBKH,"5,5-dimethyl-3-[4-nitro-3-(trifluoromethyl)phe...",CC1(C(=O)N(C(=O)N1)C2=CC(=C(C=C2)[N+](=O)[O-])...,prostate cancer,Pubchem,DOID
4,5360373,ChemicalEntity_Disease,DOID:2998,ChemicalEntity,Disease,iBKH,"3-[[2-[2-[2-[[(2S,3R)-2-[[(2S,3S,4R)-4-[[(2S,3...",CC1=C(N=C(N=C1N)[C@H](CC(=O)N)NC[C@@H](C(=O)N)...,testicular cancer,Pubchem,DOID
...,...,...,...,...,...,...,...,...,...,...,...
2717929,612,ChemicalEntity_Disease,DOID:811,ChemicalEntity,Disease,iBKH,2-hydroxypropanoic acid,CC(C(=O)O)O,lipodystrophy,Pubchem,DOID
2717930,612,ChemicalEntity_Disease,DOID:1056,ChemicalEntity,Disease,iBKH,2-hydroxypropanoic acid,CC(C(=O)O)O,oculocerebrorenal syndrome,Pubchem,DOID
2717931,16850,ChemicalEntity_Disease,DOID:8947,ChemicalEntity,Disease,iBKH,"3',6'-dihydroxyspiro[2-benzofuran-3,9'-xanthen...",C1=CC=C2C(=C1)C(=O)OC23C4=C(C=C(C=C4)O)OC5=C3C...,diabetic retinopathy,Pubchem,DOID
2717932,135398721,ChemicalEntity_Disease,DOID:150,ChemicalEntity,Disease,iBKH,"2-amino-6-[(1S,2R)-1,2,3-trihydroxypropyl]-3H-...",C1=C(N=C2C(=O)NC(=NC2=N1)N)[C@@H]([C@@H](CO)O)O,disease of mental health,Pubchem,DOID


In [156]:
# 

In [157]:
ibkh_D_Disease.to_csv(os.path.join(OUT_PATH, "Chemical_Disease_ibkh.csv"), index=False)
print("Saved: Chemical_Disease_ibkh.csv")
del ibkh_D_Disease


Saved: Chemical_Disease_ibkh.csv


## 7. Drug–Gene via HGNC IDs

In [160]:
# ibkh_Drug_Gene = pd.read_csv(os.path.join(BASE_PATH, "D_G_processed.csv"))

# prefixes_drug = ("MeSH", "Drug", "DB08", "DB01")
# ibkh_Drug_Gene = ibkh_Drug_Gene[ibkh_Drug_Gene["Head"].str.startswith(prefixes_drug, na=False)]
# ibkh_Drug_Gene["Head"] = ibkh_Drug_Gene["Head"].str.replace("^DrugBank:", "", regex=True)

# ibkh_Drug_Gene["Head_ID"] = ibkh_Drug_Gene["Head"].apply(map_drug_head_id)  
# ibkh_Drug_Gene["Head_ID"] = strip_float_suffix(ibkh_Drug_Gene["Head_ID"])  
# ibkh_Drug_Gene = ibkh_Drug_Gene[ibkh_Drug_Gene["Head_ID"].notna() & (ibkh_Drug_Gene["Head_ID"] != "nan")]

# ibkh_Drug_Gene[["Head", "Head_ID"]] = ibkh_Drug_Gene[["Head_ID", "Head"]]
# ibkh_Drug_Gene["Head_detail_name"] = ibkh_Drug_Gene["Head"].map(Pubchem_IUPAC_CID_Dict)
# ibkh_Drug_Gene["Head_SMILES_name"] = ibkh_Drug_Gene["Head"].map(Pubchem_CID_Smile_Dict)

# # Keep only HGNC Tail entries
# ibkh_Drug_Gene = ibkh_Drug_Gene[ibkh_Drug_Gene["Tail"].notna()]
# ibkh_Drug_Gene = ibkh_Drug_Gene[ibkh_Drug_Gene["Tail"].str.startswith("HGNC")]

# ibkh_Drug_Gene["Tail_ID"] = ibkh_Drug_Gene["Tail"].map(HGNC_NCBI_dict)
# ibkh_Drug_Gene[["Tail", "Tail_ID"]] = ibkh_Drug_Gene[["Tail_ID", "Tail"]]
# ibkh_Drug_Gene = ibkh_Drug_Gene[ibkh_Drug_Gene["Tail"].notna()]
# ibkh_Drug_Gene["Tail_detail_name"] = ibkh_Drug_Gene["Tail"].map(NCBI_Synbol_GENEname_dict)

# ibkh_Drug_Gene["Head_ID_IS"] = head_id_is(ibkh_Drug_Gene["Head"])
# ibkh_Drug_Gene["Tail_ID_IS"] = "NCBI_ID"
# ibkh_Drug_Gene["Head_type"]  = "ChemicalEntity"
# ibkh_Drug_Gene["Tail_type"]  = "Gene"
# ibkh_Drug_Gene["Relation"]   = ibkh_Drug_Gene["Head_type"] + "_" + ibkh_Drug_Gene["Tail_type"]
# ibkh_Drug_Gene["KG_Source"]  = "iBKH"

# ibkh_Drug_Gene = reorder(ibkh_Drug_Gene, KG_DESIRED_COLS)
# print(f"Drug–Gene (HGNC) rows: {len(ibkh_Drug_Gene):,}")
# ibkh_Drug_Gene


In [161]:
# ibkh_Drug_Gene.to_csv(os.path.join(OUT_PATH, "Drug_Gene_ibkh.csv"), index=False)
# print("Saved: Drug_Gene_ibkh.csv")
# del ibkh_Drug_Gene


## 8. Drug–Pathway Interactions

In [264]:
ibkh_Drug_Pathway = pd.read_csv(os.path.join(BASE_PATH, "ibkh/D_Pwy_res.csv"))

ibkh_Drug_Pathway  = ibkh_Drug_Pathway.rename(columns={
    "Drug": "Head",
    "Pathway": "Tail"
})

ibkh_Drug_Pathway = ibkh_Drug_Pathway[ibkh_Drug_Pathway["Head"].str.startswith("Drug", na=False)]
ibkh_Drug_Pathway["Head"] = ibkh_Drug_Pathway["Head"].str.replace("^DrugBank:", "", regex=True)

ibkh_Drug_Pathway["Head_ID"] = ibkh_Drug_Pathway["Head"].apply(map_drug_head_id)  
ibkh_Drug_Pathway["Head_ID"] = strip_float_suffix(ibkh_Drug_Pathway["Head_ID"])    
ibkh_Drug_Pathway = ibkh_Drug_Pathway[
    ibkh_Drug_Pathway["Head_ID"].notna() & (ibkh_Drug_Pathway["Head_ID"] != "nan")
]
ibkh_Drug_Pathway[["Head", "Head_ID"]] = ibkh_Drug_Pathway[["Head_ID", "Head"]]
ibkh_Drug_Pathway["Head_detail_name"] = ibkh_Drug_Pathway["Head"].map(Pubchem_IUPAC_CID_Dict).fillna(ibkh_Drug_Pathway["Head"].map(drugbank_dict))
ibkh_Drug_Pathway = ibkh_Drug_Pathway[ibkh_Drug_Pathway["Head_detail_name"].notna()]
ibkh_Drug_Pathway["Head_SMILES_name"] = ibkh_Drug_Pathway["Head"].map(Pubchem_CID_Smile_Dict)
ibkh_Drug_Pathway

,Head,Tail,Association,Source,Head_ID,Head_detail_name,Head_SMILES_name
1,2554,KEGG:map00982,1,KEGG,DB00564,benzo[b][1]benzazepine-11-carboxamide,C1=CC=C2C(=C1)C=CC3=CC=CC=C3N2C(=O)N
2,3690,KEGG:map00982,1,KEGG,DB01181,"N,3-bis(2-chloroethyl)-2-oxo-1,3,2lambda5-oxaz...",C1CN(P(=O)(OC1)NCCCl)CCCl
3,3676,KEGG:map00982,1,KEGG,DB00281,"2-(diethylamino)-N-(2,6-dimethylphenyl)acetamide",CCN(CC)CC(=O)NC1=C(C=CC=C1C)C
4,3121,KEGG:map00982,1,KEGG,DB00313,2-propylpentanoic acid,CCCC(CCC)C(=O)O
5,34312,KEGG:map00982,1,KEGG,DB00776,5-oxo-6H-benzo[b][1]benzazepine-11-carboxamide,C1C2=CC=CC=C2N(C3=CC=CC=C3C1=O)C(=O)N
...,...,...,...,...,...,...,...
3225,3689,KEGG:map07235,1,KEGG,DB08954,4-[2-(4-benzylpiperidin-1-yl)-1-hydroxypropyl]...,CC(C(C1=CC=C(C=C1)O)O)N2CCC(CC2)CC3=CC=CC=C3
3226,3821,KEGG:map07235,1,KEGG,DB01221,2-(2-chlorophenyl)-2-(methylamino)cyclohexan-1...,CNC1(CCCCC1=O)C2=CC=CC=C2Cl
3227,22267,KEGG:map07235,1,KEGG,DB13515,"(6R)-6-(dimethylamino)-4,4-diphenylheptan-3-one",CCC(=O)C(C[C@@H](C)N(C)C)(C1=CC=CC=C1)C2=CC=CC=C2
3229,4054,KEGG:map07235,1,KEGG,DB01043,"3,5-dimethyladamantan-1-amine",CC12CC3CC(C1)(CC(C3)(C2)N)C


In [265]:
ibkh_Drug_Pathway_save = ibkh_Drug_Pathway

In [266]:
# kegg_mapping_dict

In [267]:
ibkh_Drug_Pathway = ibkh_Drug_Pathway_save


In [268]:
# ibkh_Drug_Pathway["Tail"] = ibkh_Drug_Pathway["Tail"].str.replace("^KEGG:", "", regex=True)

ibkh_Drug_Pathway["Tail_Alt_ID"] = ibkh_Drug_Pathway["Tail"].map(kegg_mapping_dict)
ibkh_Drug_Pathway

,Head,Tail,Association,Source,Head_ID,Head_detail_name,Head_SMILES_name,Tail_Alt_ID
1,2554,KEGG:map00982,1,KEGG,DB00564,benzo[b][1]benzazepine-11-carboxamide,C1=CC=C2C(=C1)C=CC3=CC=CC=C3N2C(=O)N,NaN
2,3690,KEGG:map00982,1,KEGG,DB01181,"N,3-bis(2-chloroethyl)-2-oxo-1,3,2lambda5-oxaz...",C1CN(P(=O)(OC1)NCCCl)CCCl,NaN
3,3676,KEGG:map00982,1,KEGG,DB00281,"2-(diethylamino)-N-(2,6-dimethylphenyl)acetamide",CCN(CC)CC(=O)NC1=C(C=CC=C1C)C,NaN
4,3121,KEGG:map00982,1,KEGG,DB00313,2-propylpentanoic acid,CCCC(CCC)C(=O)O,NaN
5,34312,KEGG:map00982,1,KEGG,DB00776,5-oxo-6H-benzo[b][1]benzazepine-11-carboxamide,C1C2=CC=CC=C2N(C3=CC=CC=C3C1=O)C(=O)N,NaN
...,...,...,...,...,...,...,...,...
3225,3689,KEGG:map07235,1,KEGG,DB08954,4-[2-(4-benzylpiperidin-1-yl)-1-hydroxypropyl]...,CC(C(C1=CC=C(C=C1)O)O)N2CCC(CC2)CC3=CC=CC=C3,NaN
3226,3821,KEGG:map07235,1,KEGG,DB01221,2-(2-chlorophenyl)-2-(methylamino)cyclohexan-1...,CNC1(CCCCC1=O)C2=CC=CC=C2Cl,NaN
3227,22267,KEGG:map07235,1,KEGG,DB13515,"(6R)-6-(dimethylamino)-4,4-diphenylheptan-3-one",CCC(=O)C(C[C@@H](C)N(C)C)(C1=CC=CC=C1)C2=CC=CC=C2,NaN
3229,4054,KEGG:map07235,1,KEGG,DB01043,"3,5-dimethyladamantan-1-amine",CC12CC3CC(C1)(CC(C3)(C2)N)C,NaN


In [269]:
# Resolve Tail to Reactome ID where possible
ibkh_Drug_Pathway["Tail"] = (
    ibkh_Drug_Pathway["Tail_Alt_ID"].map(pathway_reactome_rev_dict)
    .fillna(ibkh_Drug_Pathway["Tail"])
)
ibkh_Drug_Pathway.rename(columns={"Tail_Alt_ID": "Tail_detail_name"}, inplace=True)

ibkh_Drug_Pathway["Head_ID_IS"] = head_id_is(ibkh_Drug_Pathway["Head"])
ibkh_Drug_Pathway["Tail_ID_IS"] = np.where(
    ibkh_Drug_Pathway["Tail"].isna(), None,
    np.where(ibkh_Drug_Pathway["Tail"].str.startswith("KEGG"), "KEGG_ID", "Reactome_ID")
)
ibkh_Drug_Pathway["Head_type"] = "ChemicalEntity"
ibkh_Drug_Pathway["Tail_type"] = "Pathway"
ibkh_Drug_Pathway["Relation"]  = ibkh_Drug_Pathway["Head_type"] + "_" + ibkh_Drug_Pathway["Tail_type"]
ibkh_Drug_Pathway["KG_Source"] = "iBKH"

ibkh_Drug_Pathway = reorder(ibkh_Drug_Pathway, KG_DESIRED_COLS + ["Tail_detail_name"])
print(f"Drug–Pathway rows: {len(ibkh_Drug_Pathway):,}")
ibkh_Drug_Pathway

Drug–Pathway rows: 1,670


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILES_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Tail_detail_name
1,2554,ChemicalEntity_Pathway,KEGG:map00982,ChemicalEntity,Pathway,iBKH,benzo[b][1]benzazepine-11-carboxamide,C1=CC=C2C(=C1)C=CC3=CC=CC=C3N2C(=O)N,NaN,Pubchem,KEGG_ID,NaN
2,3690,ChemicalEntity_Pathway,KEGG:map00982,ChemicalEntity,Pathway,iBKH,"N,3-bis(2-chloroethyl)-2-oxo-1,3,2lambda5-oxaz...",C1CN(P(=O)(OC1)NCCCl)CCCl,NaN,Pubchem,KEGG_ID,NaN
3,3676,ChemicalEntity_Pathway,KEGG:map00982,ChemicalEntity,Pathway,iBKH,"2-(diethylamino)-N-(2,6-dimethylphenyl)acetamide",CCN(CC)CC(=O)NC1=C(C=CC=C1C)C,NaN,Pubchem,KEGG_ID,NaN
4,3121,ChemicalEntity_Pathway,KEGG:map00982,ChemicalEntity,Pathway,iBKH,2-propylpentanoic acid,CCCC(CCC)C(=O)O,NaN,Pubchem,KEGG_ID,NaN
5,34312,ChemicalEntity_Pathway,KEGG:map00982,ChemicalEntity,Pathway,iBKH,5-oxo-6H-benzo[b][1]benzazepine-11-carboxamide,C1C2=CC=CC=C2N(C3=CC=CC=C3C1=O)C(=O)N,NaN,Pubchem,KEGG_ID,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3225,3689,ChemicalEntity_Pathway,KEGG:map07235,ChemicalEntity,Pathway,iBKH,4-[2-(4-benzylpiperidin-1-yl)-1-hydroxypropyl]...,CC(C(C1=CC=C(C=C1)O)O)N2CCC(CC2)CC3=CC=CC=C3,NaN,Pubchem,KEGG_ID,NaN
3226,3821,ChemicalEntity_Pathway,KEGG:map07235,ChemicalEntity,Pathway,iBKH,2-(2-chlorophenyl)-2-(methylamino)cyclohexan-1...,CNC1(CCCCC1=O)C2=CC=CC=C2Cl,NaN,Pubchem,KEGG_ID,NaN
3227,22267,ChemicalEntity_Pathway,KEGG:map07235,ChemicalEntity,Pathway,iBKH,"(6R)-6-(dimethylamino)-4,4-diphenylheptan-3-one",CCC(=O)C(C[C@@H](C)N(C)C)(C1=CC=CC=C1)C2=CC=CC=C2,NaN,Pubchem,KEGG_ID,NaN
3229,4054,ChemicalEntity_Pathway,KEGG:map07235,ChemicalEntity,Pathway,iBKH,"3,5-dimethyladamantan-1-amine",CC12CC3CC(C1)(CC(C3)(C2)N)C,NaN,Pubchem,KEGG_ID,NaN


In [270]:
ibkh_Drug_Pathway.to_csv(os.path.join(OUT_PATH, "Drug_Pathway_ibkh.csv"), index=False)
print("Saved: Drug_Pathway_ibkh.csv")
del ibkh_Drug_Pathway

Saved: Drug_Pathway_ibkh.csv


## 9. Drug–Side-Effect


In [ ]:
# ibkh_Drug_SE = pd.read_csv(os.path.join(BASE_PATH, "D_SE_processed.csv"))

# # Resolve UMLS CUI → HPO ID (via MedGen)
# ibkh_Drug_SE["Tail"] = ibkh_Drug_SE["Tail"].str.replace("^UMLS:", "", regex=True)
# ibkh_Drug_SE["Tail_ID"] = ibkh_Drug_SE["Tail"].map(MedGen_CUID_Source_ID_dict)
# ibkh_Drug_SE = ibkh_Drug_SE[ibkh_Drug_SE["Tail_ID"].notna()]
# ibkh_Drug_SE[["Tail", "Tail_ID"]] = ibkh_Drug_SE[["Tail_ID", "Tail"]]
# ibkh_Drug_SE["Tail_detail_name"] = ibkh_Drug_SE["Tail"].map(HPOphenotype_name_dict)
# ibkh_Drug_SE = ibkh_Drug_SE[ibkh_Drug_SE["Tail_detail_name"].notna()]

# prefixes_drug = ("MeSH", "Drug", "DB08", "DB01")
# ibkh_Drug_SE = ibkh_Drug_SE[ibkh_Drug_SE["Head"].str.startswith(prefixes_drug, na=False)]
# ibkh_Drug_SE["Head"] = ibkh_Drug_SE["Head"].str.replace("^DrugBank:", "", regex=True)

# ibkh_Drug_SE["Head_ID"] = ibkh_Drug_SE["Head"].apply(map_drug_head_id)  
# ibkh_Drug_SE["Head_ID"] = strip_float_suffix(ibkh_Drug_SE["Head_ID"])  
# ibkh_Drug_SE = ibkh_Drug_SE[ibkh_Drug_SE["Head_ID"].notna() & (ibkh_Drug_SE["Head_ID"] != "nan")]

# ibkh_Drug_SE[["Head", "Head_ID"]] = ibkh_Drug_SE[["Head_ID", "Head"]]
# ibkh_Drug_SE["Head_detail_name"] = ibkh_Drug_SE["Head"].map(Pubchem_IUPAC_CID_Dict)
# ibkh_Drug_SE["Head_SMILES_name"] = ibkh_Drug_SE["Head"].map(Pubchem_CID_Smile_Dict)

# ibkh_Drug_SE["Head_ID_IS"] = head_id_is(ibkh_Drug_SE["Head"])
# ibkh_Drug_SE["Tail_ID_IS"] = "HPO_ID"
# ibkh_Drug_SE["Head_type"]  = "ChemicalEntity"
# ibkh_Drug_SE["Tail_type"]  = "SideEffect"
# ibkh_Drug_SE["Relation"]   = ibkh_Drug_SE["Head_type"] + "_" + ibkh_Drug_SE["Tail_type"]
# ibkh_Drug_SE["KG_Source"]  = "iBKH"

# ibkh_Drug_SE = reorder(ibkh_Drug_SE, KG_DESIRED_COLS)
# print(f"Drug–SideEffect rows: {len(ibkh_Drug_SE):,}")
# ibkh_Drug_SE


In [ ]:
# ibkh_Drug_SE.to_csv(os.path.join(OUT_PATH, "Chemical_Sideffect_ibkh.csv"), index=False)
# print("Saved: Chemical_Sideffect_ibkh.csv")
# del ibkh_Drug_SE


## 10. Disease–Disease Relationships

In [185]:
ibkh_Disease_dis = pd.read_csv(os.path.join(BASE_PATH, "ibkh/Di_Di_res.csv"))

ibkh_Disease_dis  = ibkh_Disease_dis.rename(columns={
    "Disease_1": "Head",
    "Disease_2": "Tail"
})
ibkh_Disease_dis
ibkh_Disease_dis["Head_detail_name"] = ibkh_Disease_dis["Head"].map(DOID_disname_dict)
ibkh_Disease_dis["Tail_detail_name"] = ibkh_Disease_dis["Tail"].map(DOID_disname_dict)
ibkh_Disease_dis = ibkh_Disease_dis[ibkh_Disease_dis["Head_detail_name"].notna()]
ibkh_Disease_dis = ibkh_Disease_dis[ibkh_Disease_dis["Tail_detail_name"].notna()]

# De-duplicate columns if any
ibkh_Disease_dis = ibkh_Disease_dis.loc[:, ~ibkh_Disease_dis.columns.duplicated(keep="last")]

ibkh_Disease_dis["Head_ID_IS"] = "DOID"
ibkh_Disease_dis["Tail_ID_IS"] = "DOID"
ibkh_Disease_dis["Head_type"]  = "Disease"
ibkh_Disease_dis["Tail_type"]  = "Disease"
ibkh_Disease_dis["Relation"]   = ibkh_Disease_dis["Head_type"] + "_" + ibkh_Disease_dis["Tail_type"]
ibkh_Disease_dis["KG_Source"]  = "iBKH"

ibkh_Disease_dis = reorder(ibkh_Disease_dis, KG_DESIRED_COLS)
print(f"Disease–Disease rows: {len(ibkh_Disease_dis):,}")
ibkh_Disease_dis


Disease–Disease rows: 11,022


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,DOID:0001816,Disease_Disease,DOID:175,Disease,Disease,iBKH,angiosarcoma,vascular cancer,DOID,DOID
1,DOID:175,Disease_Disease,DOID:176,Disease,Disease,iBKH,vascular cancer,cardiovascular cancer,DOID,DOID
2,DOID:0002116,Disease_Disease,DOID:10124,Disease,Disease,iBKH,pterygium,corneal disease,DOID,DOID
3,DOID:10124,Disease_Disease,DOID:5614,Disease,Disease,iBKH,corneal disease,eye disease,DOID,DOID
4,DOID:0014667,Disease_Disease,DOID:4,Disease,Disease,iBKH,disease of metabolism,disease,DOID,DOID
...,...,...,...,...,...,...,...,...,...,...
11067,DOID:219,Disease_Disease,DOID:8577,Disease,Disease,iBKH,colon cancer,ulcerative colitis,DOID,DOID
11068,DOID:2994,Disease_Disease,DOID:13499,Disease,Disease,iBKH,germ cell cancer,jejunal cancer,DOID,DOID
11069,DOID:1793,Disease_Disease,DOID:10534,Disease,Disease,iBKH,pancreatic cancer,stomach cancer,DOID,DOID
11070,DOID:219,Disease_Disease,DOID:3121,Disease,Disease,iBKH,colon cancer,gallbladder cancer,DOID,DOID


In [186]:
ibkh_Disease_dis.to_csv(os.path.join(OUT_PATH, "Disease_Disease_ibkh.csv"), index=False)
print("Saved: Disease_Disease_ibkh.csv")
del ibkh_Disease_dis


Saved: Disease_Disease_ibkh.csv


## 11. Disease–Gene Associations


In [202]:
# ibkh_Disease_Gene = pd.read_csv(os.path.join(BASE_PATH, "Di_G_processed.csv"))
ibkh_Disease_Gene = pd.read_csv(os.path.join(BASE_PATH, "ibkh/Di_G_res.csv"))

ibkh_Disease_Gene  = ibkh_Disease_Gene.rename(columns={
    "Disease": "Head",
    "Gene": "Tail"
})
# Keep DOID or MeSH heads; HGNC or NCBI tails
ibkh_Disease_Gene = ibkh_Disease_Gene[
    ibkh_Disease_Gene["Head"].str.startswith(("DOID", "MeSH"), na=False)
]
ibkh_Disease_Gene = ibkh_Disease_Gene[
    ibkh_Disease_Gene["Tail"].str.startswith(("HGNC", "NCBI"), na=False)
]

# Tail: resolve to Gene symbol
ibkh_Disease_Gene["Tail"] = ibkh_Disease_Gene["Tail"].str.replace("^NCBI:", "", regex=True)
ibkh_Disease_Gene["Tail_ID"] = (
    ibkh_Disease_Gene["Tail"].map(HGNC_NCBI_dict)
    .fillna(ibkh_Disease_Gene["Tail"].map(NCBI_ALL_GENEIDname_dict))
)
ibkh_Disease_Gene[["Tail", "Tail_ID"]] = ibkh_Disease_Gene[["Tail_ID", "Tail"]]
ibkh_Disease_Gene = ibkh_Disease_Gene[ibkh_Disease_Gene["Tail"].notna()]
ibkh_Disease_Gene["Tail_detail_name"] = ibkh_Disease_Gene["Tail"].map(NCBI_Synbol_GENEname_dict)

# Head: normalise MeSH prefix, then map to DOID
ibkh_Disease_Gene["Head"] = ibkh_Disease_Gene["Head"].apply(
    lambda x: x.replace("MeSH:", "MESH:") if isinstance(x, str) and x.startswith("MeSH:") else x
)
ibkh_Disease_Gene["Head_ID"] = ibkh_Disease_Gene["Head"].map(Mesh2DOID_dict).fillna(ibkh_Disease_Gene["Head"])
ibkh_Disease_Gene[["Head", "Head_ID"]] = ibkh_Disease_Gene[["Head_ID", "Head"]]
ibkh_Disease_Gene = ibkh_Disease_Gene[ibkh_Disease_Gene["Head"].notna()]

# Strip MESH: prefix then annotate
ibkh_Disease_Gene["Head"] = ibkh_Disease_Gene["Head"].apply(
    lambda x: x.replace("MeSH:", "MESH:") if isinstance(x, str) and x.startswith("MeSH:") else x
)
ibkh_Disease_Gene["Head"] = ibkh_Disease_Gene["Head"].str.replace("^MESH:", "", regex=True)
ibkh_Disease_Gene["Head_detail_name"] = (
    ibkh_Disease_Gene["Head"].map(DOID_disname_dict)
    .fillna(ibkh_Disease_Gene["Head"].map(Mesh_supp_dict))
)
ibkh_Disease_Gene = ibkh_Disease_Gene[ibkh_Disease_Gene["Head_detail_name"].notna()]

ibkh_Disease_Gene["Head_ID_IS"] = np.where(
    ibkh_Disease_Gene["Head"].isna(), None,
    np.where(ibkh_Disease_Gene["Head"].str.startswith("DOID"), "DOID", "MESH")
)
ibkh_Disease_Gene["Tail_ID_IS"] = "NCBI_ID"
ibkh_Disease_Gene["Head_type"]  = "Disease"
ibkh_Disease_Gene["Tail_type"]  = "Gene"

ibkh_Disease_Gene["Relation"]   = ibkh_Disease_Gene["Head_type"] + "_" + ibkh_Disease_Gene["Tail_type"]
ibkh_Disease_Gene["KG_Source"]  = "iBKH"

ibkh_Disease_Gene = reorder(ibkh_Disease_Gene, KG_DESIRED_COLS)
print(f"Disease–Gene rows: {len(ibkh_Disease_Gene):,}")
ibkh_Disease_Gene

Disease–Gene rows: 12,323,320


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,DOID:263,Disease_Gene,IFNA1,Disease,Gene,iBKH,kidney cancer,interferon alpha 1,DOID,NCBI_ID
1,DOID:1909,Disease_Gene,VCAN,Disease,Gene,iBKH,melanoma,versican,DOID,NCBI_ID
2,DOID:10763,Disease_Gene,CRHR1,Disease,Gene,iBKH,hypertension,corticotropin releasing hormone receptor 1,DOID,NCBI_ID
3,DOID:2394,Disease_Gene,MMP2,Disease,Gene,iBKH,ovarian cancer,matrix metallopeptidase 2,DOID,NCBI_ID
4,DOID:12849,Disease_Gene,OXTR,Disease,Gene,iBKH,autistic disorder,oxytocin receptor,DOID,NCBI_ID
...,...,...,...,...,...,...,...,...,...,...
27538756,DOID:4674,Disease_Gene,TNF,Disease,Gene,iBKH,androgen insensitivity syndrome,tumor necrosis factor,DOID,NCBI_ID
27538757,DOID:14755,Disease_Gene,TP53,Disease,Gene,iBKH,argininosuccinic aciduria,tumor protein p53,DOID,NCBI_ID
27538758,DOID:0110078,Disease_Gene,VDR,Disease,Gene,iBKH,Leber congenital amaurosis 1,vitamin D receptor,DOID,NCBI_ID
27538764,DOID:0060299,Disease_Gene,CALM1,Disease,Gene,iBKH,complement component 6 deficiency,calmodulin 1,DOID,NCBI_ID


In [203]:
ibkh_Disease_Gene.to_csv(os.path.join(OUT_PATH, "Disease_Gene_ibkh.csv"), index=False)
print("Saved: Disease_Gene_ibkh.csv")
del ibkh_Disease_Gene


Saved: Disease_Gene_ibkh.csv


## 12. Gene–Gene Interactions


In [206]:
ibkh_Gene_Gene = pd.read_csv(os.path.join(BASE_PATH, "ibkh/G_G_res.csv"))

ibkh_Gene_Gene  = ibkh_Gene_Gene.rename(columns={
    "Gene_1": "Head",
    "Gene_2": "Tail"
})
ibkh_Gene_Gene
ibkh_Gene_Gene = ibkh_Gene_Gene[ibkh_Gene_Gene["Head"].str.startswith(("HGNC", "NCBI"), na=False)]
ibkh_Gene_Gene = ibkh_Gene_Gene[ibkh_Gene_Gene["Tail"].str.startswith(("HGNC", "NCBI"), na=False)]

for col in ("Head", "Tail"):
    ibkh_Gene_Gene[col] = ibkh_Gene_Gene[col].str.replace("^NCBI:", "", regex=True)
    ibkh_Gene_Gene[f"{col}_ID"] = (
        ibkh_Gene_Gene[col].map(HGNC_NCBI_dict)
        .fillna(ibkh_Gene_Gene[col].map(NCBI_ALL_GENEIDname_dict))
    )
    ibkh_Gene_Gene[[col, f"{col}_ID"]] = ibkh_Gene_Gene[[f"{col}_ID", col]]
    ibkh_Gene_Gene = ibkh_Gene_Gene[ibkh_Gene_Gene[col].notna()]
    ibkh_Gene_Gene[f"{col}_detail_name"] = ibkh_Gene_Gene[col].map(NCBI_Synbol_GENEname_dict)

ibkh_Gene_Gene["Head_ID_IS"] = "NCBI_ID"
ibkh_Gene_Gene["Tail_ID_IS"] = "NCBI_ID"
ibkh_Gene_Gene["Head_type"]  = "Gene"
ibkh_Gene_Gene["Tail_type"]  = "Gene"
ibkh_Gene_Gene["Relation"]   = ibkh_Gene_Gene["Head_type"] + "_" + ibkh_Gene_Gene["Tail_type"]
ibkh_Gene_Gene["KG_Source"]  = "iBKH"

ibkh_Gene_Gene = reorder(ibkh_Gene_Gene, KG_DESIRED_COLS)
print(f"Gene–Gene rows: {len(ibkh_Gene_Gene):,}")
ibkh_Gene_Gene


Gene–Gene rows: 711,649


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,IMP3,Gene_Gene,OR8U8,Gene,Gene,iBKH,IMP U3 small nucleolar ribonucleoprotein 3,olfactory receptor family 8 subfamily U member 8,NCBI_ID,NCBI_ID
1,FADD,Gene_Gene,C1orf56,Gene,Gene,iBKH,Fas associated via death domain,chromosome 1 open reading frame 56,NCBI_ID,NCBI_ID
2,TRABD2B,Gene_Gene,IRX1,Gene,Gene,iBKH,TraB domain containing 2B,iroquois homeobox 1,NCBI_ID,NCBI_ID
3,OPN1LW,Gene_Gene,ZDHHC16,Gene,Gene,iBKH,"opsin 1, long wave sensitive",zinc finger DHHC-type palmitoyltransferase 16,NCBI_ID,NCBI_ID
4,TENM1,Gene_Gene,FKBP1B,Gene,Gene,iBKH,teneurin transmembrane protein 1,FKBP prolyl isomerase 1B,NCBI_ID,NCBI_ID
...,...,...,...,...,...,...,...,...,...,...
735151,ADAMTS12,Gene_Gene,A2M,Gene,Gene,iBKH,ADAM metallopeptidase with thrombospondin type...,alpha-2-macroglobulin,NCBI_ID,NCBI_ID
735152,CASP3,Gene_Gene,ATG16L1,Gene,Gene,iBKH,caspase 3,autophagy related 16 like 1,NCBI_ID,NCBI_ID
735153,CASP4,Gene_Gene,GSDMD,Gene,Gene,iBKH,caspase 4,gasdermin D,NCBI_ID,NCBI_ID
735154,CASP6,Gene_Gene,ATG16L1,Gene,Gene,iBKH,caspase 6,autophagy related 16 like 1,NCBI_ID,NCBI_ID


In [207]:
ibkh_Gene_Gene.to_csv(os.path.join(OUT_PATH, "Gene_Gene_ibkh.csv"), index=False)
print("Saved: Gene_Gene_ibkh.csv")
del ibkh_Gene_Gene


Saved: Gene_Gene_ibkh.csv


## 13. Gene–Tissue (BTO) from Anatomy file

In [217]:
ibkh_Anatomy_gene = pd.read_csv(os.path.join(BASE_PATH, "ibkh/A_G_res.csv"))

ibkh_Anatomy_gene  = ibkh_Anatomy_gene.rename(columns={
    "Gene": "Head",
    "Anatomy": "Tail"
})
ibkh_Anatomy_gene = ibkh_Anatomy_gene[
    ibkh_Anatomy_gene["Head"].str.startswith(("HGNC", "NCBI"), na=False)
]

ibkh_Anatomy_gene["Head"] = ibkh_Anatomy_gene["Head"].str.replace("^NCBI:", "", regex=True)
ibkh_Anatomy_gene["Head_ID"] = (
    ibkh_Anatomy_gene["Head"].map(HGNC_NCBI_dict)
    .fillna(ibkh_Anatomy_gene["Head"].map(NCBI_ALL_GENEIDname_dict))
)
ibkh_Anatomy_gene[["Head", "Head_ID"]] = ibkh_Anatomy_gene[["Head_ID", "Head"]]
ibkh_Anatomy_gene = ibkh_Anatomy_gene[ibkh_Anatomy_gene["Head"].notna()]
ibkh_Anatomy_gene["Head_detail_name"] = ibkh_Anatomy_gene["Head"].map(NCBI_Synbol_GENEname_dict)
ibkh_Anatomy_gene

,Head,Tail,Express,Absent,Source,Head_ID,Head_detail_name
0,TSPAN6,UBERON:0000002,1,0,Bgee;TISSUE,HGNC:11858,tetraspanin 6
1,TSPAN6,UBERON:0000006,1,0,Bgee,HGNC:11858,tetraspanin 6
2,TSPAN6,UBERON:0000007,1,0,Bgee,HGNC:11858,tetraspanin 6
3,TSPAN6,UBERON:0000014,1,0,Bgee,HGNC:11858,tetraspanin 6
4,TSPAN6,UBERON:0000029,1,0,Bgee;TISSUE,HGNC:11858,tetraspanin 6
...,...,...,...,...,...,...,...
12170971,ZSCAN16-AS1,BTO:0000876,1,0,TISSUE,HGNC:48982,ZSCAN16 antisense RNA 1
12170972,ZSCAN16-AS1,BTO:0002741,1,0,TISSUE,HGNC:48982,ZSCAN16 antisense RNA 1
12170973,ZSWIM8-AS1,BTO:0000362,1,0,TISSUE,HGNC:45103,ZSWIM8 antisense RNA 1
12170974,ZSWIM8-AS1,UBERON:0000341,1,0,TISSUE,HGNC:45103,ZSWIM8 antisense RNA 1


In [221]:
# Gene–Tissue (BTO ontology)
ibkh_Gene_Tissue = ibkh_Anatomy_gene[ibkh_Anatomy_gene["Tail"].str.startswith("BTO")].copy()
ibkh_Gene_Tissue['Tail_Alt_ID'] = ibkh_Gene_Tissue['Tail'].map(bto_tissue_dict)
ibkh_Gene_Tissue.rename(columns={"Tail_Alt_ID": "Tail_detail_name"}, inplace=True)

ibkh_Gene_Tissue["Head_ID_IS"] = "NCBI_ID"
ibkh_Gene_Tissue["Tail_ID_IS"] = "BTO_ID"
ibkh_Gene_Tissue["Head_type"]  = "Gene"
ibkh_Gene_Tissue["Tail_type"]  = "Tissue"
ibkh_Gene_Tissue["Relation"]   = "Gene_Tissue"
ibkh_Gene_Tissue["KG_Source"]  = "iBKH"

ibkh_Gene_Tissue = reorder(ibkh_Gene_Tissue, KG_DESIRED_COLS + ["Tail_detail_name"])
print(f"Gene–Tissue rows: {len(ibkh_Gene_Tissue):,}")
ibkh_Gene_Tissue

Gene–Tissue rows: 3,041,850


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Tail_detail_name
6486976,A1BG-AS1,Gene_Tissue,BTO:0000608,Gene,Tissue,iBKH,A1BG antisense RNA 1,hepatoma cell,NCBI_ID,BTO_ID,hepatoma cell
6486977,A1BG-AS1,Gene_Tissue,BTO:0000594,Gene,Tissue,iBKH,A1BG antisense RNA 1,liver cancer cell,NCBI_ID,BTO_ID,liver cancer cell
6486978,A1BG-AS1,Gene_Tissue,BTO:0000599,Gene,Tissue,iBKH,A1BG antisense RNA 1,Hep-G2 cell,NCBI_ID,BTO_ID,Hep-G2 cell
6486979,A1BG-AS1,Gene_Tissue,BTO:0000578,Gene,Tissue,iBKH,A1BG antisense RNA 1,hepatoma cell line,NCBI_ID,BTO_ID,hepatoma cell line
6486980,A1BG-AS1,Gene_Tissue,BTO:0006081,Gene,Tissue,iBKH,A1BG antisense RNA 1,hiPSC cell,NCBI_ID,BTO_ID,hiPSC cell
...,...,...,...,...,...,...,...,...,...,...,...
12170968,ZRANB2-DT,Gene_Tissue,BTO:0000930,Gene,Tissue,iBKH,ZRANB2 divergent transcript,neuroblast,NCBI_ID,BTO_ID,neuroblast
12170970,ZSCAN16-AS1,Gene_Tissue,BTO:0001251,Gene,Tissue,iBKH,ZSCAN16 antisense RNA 1,liver sinusoidal endothelial cell,NCBI_ID,BTO_ID,liver sinusoidal endothelial cell
12170971,ZSCAN16-AS1,Gene_Tissue,BTO:0000876,Gene,Tissue,iBKH,ZSCAN16 antisense RNA 1,monocyte,NCBI_ID,BTO_ID,monocyte
12170972,ZSCAN16-AS1,Gene_Tissue,BTO:0002741,Gene,Tissue,iBKH,ZSCAN16 antisense RNA 1,hepatic stellate cell,NCBI_ID,BTO_ID,hepatic stellate cell


In [222]:
ibkh_Gene_Tissue.to_csv(os.path.join(OUT_PATH, "Gene_Tissue_ibkh.csv"), index=False)
print("Saved: Gene_Tissue_ibkh.csv")
del ibkh_Gene_Tissue


Saved: Gene_Tissue_ibkh.csv


## 14. Gene–Anatomy (UBERON) from Anatomy file

In [227]:
# ibkh_Anatomy_gene already filtered above in Section 13
ibkh_Gene_Anatomy = ibkh_Anatomy_gene[ibkh_Anatomy_gene["Tail"].str.startswith("UBE")].copy()
ibkh_Gene_Anatomy['Tail_Alt_ID'] = ibkh_Gene_Anatomy['Tail'].map(uberon_dict)


ibkh_Gene_Anatomy.rename(columns={"Tail_Alt_ID": "Tail_detail_name"}, inplace=True)

ibkh_Gene_Anatomy["Head_ID_IS"] = "NCBI_ID"
ibkh_Gene_Anatomy["Tail_ID_IS"] = "UBERON_ID"
ibkh_Gene_Anatomy["Head_type"]  = "Gene"
ibkh_Gene_Anatomy["Tail_type"]  = "Anatomy"
ibkh_Gene_Anatomy["Relation"]   = "Gene_Anatomy"
ibkh_Gene_Anatomy["KG_Source"]  = "iBKH"

ibkh_Gene_Anatomy = reorder(ibkh_Gene_Anatomy, KG_DESIRED_COLS + ["Tail_detail_name"])
print(f"Gene–Anatomy rows: {len(ibkh_Gene_Anatomy):,}")
ibkh_Gene_Anatomy


Gene–Anatomy rows: 8,769,059


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Tail_detail_name
0,TSPAN6,Gene_Anatomy,UBERON:0000002,Gene,Anatomy,iBKH,tetraspanin 6,uterine cervix,NCBI_ID,UBERON_ID,uterine cervix
1,TSPAN6,Gene_Anatomy,UBERON:0000006,Gene,Anatomy,iBKH,tetraspanin 6,islet of Langerhans,NCBI_ID,UBERON_ID,islet of Langerhans
2,TSPAN6,Gene_Anatomy,UBERON:0000007,Gene,Anatomy,iBKH,tetraspanin 6,pituitary gland,NCBI_ID,UBERON_ID,pituitary gland
3,TSPAN6,Gene_Anatomy,UBERON:0000014,Gene,Anatomy,iBKH,tetraspanin 6,zone of skin,NCBI_ID,UBERON_ID,zone of skin
4,TSPAN6,Gene_Anatomy,UBERON:0000029,Gene,Anatomy,iBKH,tetraspanin 6,lymph node,NCBI_ID,UBERON_ID,lymph node
...,...,...,...,...,...,...,...,...,...,...,...
12170964,ZRANB2-DT,Gene_Anatomy,UBERON:0000451,Gene,Anatomy,iBKH,ZRANB2 divergent transcript,prefrontal cortex,NCBI_ID,UBERON_ID,prefrontal cortex
12170966,ZRANB2-DT,Gene_Anatomy,UBERON:0001870,Gene,Anatomy,iBKH,ZRANB2 divergent transcript,frontal cortex,NCBI_ID,UBERON_ID,frontal cortex
12170969,ZRANB2-DT,Gene_Anatomy,UBERON:0016526,Gene,Anatomy,iBKH,ZRANB2 divergent transcript,lobe of cerebral hemisphere,NCBI_ID,UBERON_ID,lobe of cerebral hemisphere
12170974,ZSWIM8-AS1,Gene_Anatomy,UBERON:0000341,Gene,Anatomy,iBKH,ZSWIM8 antisense RNA 1,throat,NCBI_ID,UBERON_ID,throat


In [228]:
ibkh_Gene_Anatomy.to_csv(os.path.join(OUT_PATH, "Gene_Anatomy_ibkh.csv"), index=False)
print("Saved: Gene_Anatomy_ibkh.csv")
del ibkh_Gene_Anatomy, ibkh_Anatomy_gene


Saved: Gene_Anatomy_ibkh.csv


## 15. Gene–Pathway


In [274]:
ibkh_Gene_Pathway = pd.read_csv(os.path.join(BASE_PATH, "ibkh/G_Pwy_res.csv"))

ibkh_Gene_Pathway  = ibkh_Gene_Pathway.rename(columns={
    "Gene": "Head",
    "Pathway": "Tail"
})
ibkh_Gene_Pathwayibkh_Gene_Pathway = ibkh_Gene_Pathway[
    ibkh_Gene_Pathway["Head"].str.startswith(("HGNC", "NCBI"), na=False)
]

ibkh_Gene_Pathway["Head"] = ibkh_Gene_Pathway["Head"].str.replace("^NCBI:", "", regex=True)
ibkh_Gene_Pathway["Head_ID"] = (
    ibkh_Gene_Pathway["Head"].map(HGNC_NCBI_dict)
    .fillna(ibkh_Gene_Pathway["Head"].map(NCBI_ALL_GENEIDname_dict))
)
ibkh_Gene_Pathway[["Head", "Head_ID"]] = ibkh_Gene_Pathway[["Head_ID", "Head"]]
ibkh_Gene_Pathway = ibkh_Gene_Pathway[ibkh_Gene_Pathway["Head"].notna()]
ibkh_Gene_Pathway["Head_detail_name"] = ibkh_Gene_Pathway["Head"].map(NCBI_Synbol_GENEname_dict)
ibkh_Gene_Pathway


ibkh_Gene_Pathway["Tail"] = ibkh_Gene_Pathway["Tail"].str.replace("^KEGG:", "", regex=True)
ibkh_Gene_Pathway["Tail"] = ibkh_Gene_Pathway["Tail"].str.replace("^REACT:", "", regex=True)


ibkh_Gene_Pathway["Tail_Alt_ID"] = ibkh_Gene_Pathway["Tail"].map(kegg_mapping_dict).fillna(ibkh_Gene_Pathway["Tail"].map(pathway_reactome_dict))
ibkh_Gene_Pathway


,Head,Tail,Reaction,Associate,Source,Head_ID,Head_detail_name,Tail_Alt_ID
0,A1BG,R-HSA-109582,1,0,Reactome,HGNC:5,alpha-1-B glycoprotein,Hemostasis
1,A1BG,R-HSA-114608,1,0,Reactome,HGNC:5,alpha-1-B glycoprotein,Platelet degranulation
2,A1BG,R-HSA-168249,1,0,Reactome,HGNC:5,alpha-1-B glycoprotein,Innate Immune System
3,A1BG,R-HSA-168256,1,0,Reactome,HGNC:5,alpha-1-B glycoprotein,Immune System
4,A1BG,R-HSA-6798695,1,0,Reactome,HGNC:5,alpha-1-B glycoprotein,Neutrophil degranulation
...,...,...,...,...,...,...,...,...
152238,CALML4,map05418,0,1,KEGG,HGNC:18445,calmodulin like 4,Fluid shear stress and atherosclerosis
152239,ACVR2A,map05418,0,1,KEGG,HGNC:173,activin A receptor type 2A,Fluid shear stress and atherosclerosis
152240,ACVR2B,map05418,0,1,KEGG,HGNC:174,activin A receptor type 2B,Fluid shear stress and atherosclerosis
152241,GSTO1,map05418,0,1,KEGG,HGNC:13312,glutathione S-transferase omega 1,Fluid shear stress and atherosclerosis


In [275]:

ibkh_Gene_Pathway.rename(columns={"Tail_Alt_ID": "Tail_detail_name"}, inplace=True)
ibkh_Gene_Pathway["Head_ID_IS"] = "NCBI_ID"
ibkh_Gene_Pathway["Tail_ID_IS"] = np.where(
    ibkh_Gene_Pathway["Tail"].isna(), None,
    np.where(
        ibkh_Gene_Pathway["Tail"].str.startswith("R-HSA"), "Reactome_ID",
        np.where(ibkh_Gene_Pathway["Tail"].str.startswith("KEGG"), "KEGG", None)
    )
)
ibkh_Gene_Pathway["Head_type"] = "Gene"
ibkh_Gene_Pathway["Tail_type"] = "Pathway"
ibkh_Gene_Pathway["Relation"]  = ibkh_Gene_Pathway["Head_type"] + "_" + ibkh_Gene_Pathway["Tail_type"]
ibkh_Gene_Pathway["KG_Source"] = "iBKH"

ibkh_Gene_Pathway = reorder(ibkh_Gene_Pathway, KG_DESIRED_COLS + ["Tail_detail_name"])
print(f"Gene–Pathway rows: {len(ibkh_Gene_Pathway):,}")
ibkh_Gene_Pathway


Gene–Pathway rows: 150,367


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Tail_detail_name
0,A1BG,Gene_Pathway,R-HSA-109582,Gene,Pathway,iBKH,alpha-1-B glycoprotein,Hemostasis,NCBI_ID,Reactome_ID,Hemostasis
1,A1BG,Gene_Pathway,R-HSA-114608,Gene,Pathway,iBKH,alpha-1-B glycoprotein,Platelet degranulation,NCBI_ID,Reactome_ID,Platelet degranulation
2,A1BG,Gene_Pathway,R-HSA-168249,Gene,Pathway,iBKH,alpha-1-B glycoprotein,Innate Immune System,NCBI_ID,Reactome_ID,Innate Immune System
3,A1BG,Gene_Pathway,R-HSA-168256,Gene,Pathway,iBKH,alpha-1-B glycoprotein,Immune System,NCBI_ID,Reactome_ID,Immune System
4,A1BG,Gene_Pathway,R-HSA-6798695,Gene,Pathway,iBKH,alpha-1-B glycoprotein,Neutrophil degranulation,NCBI_ID,Reactome_ID,Neutrophil degranulation
...,...,...,...,...,...,...,...,...,...,...,...
152238,CALML4,Gene_Pathway,map05418,Gene,Pathway,iBKH,calmodulin like 4,Fluid shear stress and atherosclerosis,NCBI_ID,None,Fluid shear stress and atherosclerosis
152239,ACVR2A,Gene_Pathway,map05418,Gene,Pathway,iBKH,activin A receptor type 2A,Fluid shear stress and atherosclerosis,NCBI_ID,None,Fluid shear stress and atherosclerosis
152240,ACVR2B,Gene_Pathway,map05418,Gene,Pathway,iBKH,activin A receptor type 2B,Fluid shear stress and atherosclerosis,NCBI_ID,None,Fluid shear stress and atherosclerosis
152241,GSTO1,Gene_Pathway,map05418,Gene,Pathway,iBKH,glutathione S-transferase omega 1,Fluid shear stress and atherosclerosis,NCBI_ID,None,Fluid shear stress and atherosclerosis


In [276]:
ibkh_Gene_Pathway.to_csv(os.path.join(OUT_PATH, "Gene_Pathway_ibkh.csv"), index=False)
print("Saved: Gene_Pathway_ibkh.csv")
del ibkh_Gene_Pathway


Saved: Gene_Pathway_ibkh.csv


# 16 Disease Pathway

In [271]:
ibkh_Disease_Pathway = pd.read_csv(os.path.join(BASE_PATH, "ibkh/Di_Pwy_res.csv"))
ibkh_Disease_Pathway  = ibkh_Disease_Pathway.rename(columns={
    "Disease": "Head",
    "Pathway": "Tail"
})
ibkh_Disease_Pathway

# Keep DOID or MeSH heads; HGNC or NCBI tails
ibkh_Disease_Pathway = ibkh_Disease_Pathway[
    ibkh_Disease_Pathway["Head"].str.startswith(("DOID", "MeSH"), na=False)
]

# Head: normalise MeSH prefix, then map to DOID
ibkh_Disease_Pathway["Head"] = ibkh_Disease_Pathway["Head"].apply(
    lambda x: x.replace("MeSH:", "MESH:") if isinstance(x, str) and x.startswith("MeSH:") else x
)
ibkh_Disease_Pathway["Head_ID"] = ibkh_Disease_Pathway["Head"].map(Mesh2DOID_dict).fillna(ibkh_Disease_Pathway["Head"])
ibkh_Disease_Pathway[["Head", "Head_ID"]] = ibkh_Disease_Pathway[["Head_ID", "Head"]]
ibkh_Disease_Pathway = ibkh_Disease_Pathway[ibkh_Disease_Pathway["Head"].notna()]

# Strip MESH: prefix then annotate
ibkh_Disease_Pathway["Head"] = ibkh_Disease_Pathway["Head"].apply(
    lambda x: x.replace("MeSH:", "MESH:") if isinstance(x, str) and x.startswith("MeSH:") else x
)
ibkh_Disease_Pathway["Head"] = ibkh_Disease_Pathway["Head"].str.replace("^MESH:", "", regex=True)
ibkh_Disease_Pathway["Head_detail_name"] = (
    ibkh_Disease_Pathway["Head"].map(DOID_disname_dict)
    .fillna(ibkh_Disease_Pathway["Head"].map(Mesh_supp_dict))
)
ibkh_Disease_Pathway = ibkh_Disease_Pathway[ibkh_Disease_Pathway["Head_detail_name"].notna()]

ibkh_Disease_Pathway["Head_ID_IS"] = np.where(
    ibkh_Disease_Pathway["Head"].isna(), None,
    np.where(ibkh_Disease_Pathway["Head"].str.startswith("DOID"), "DOID", "MESH")
)


ibkh_Disease_Pathway["Tail_Alt_ID"] = ibkh_Disease_Pathway["Tail"].map(kegg_pathway_dict)

ibkh_Disease_Pathway.rename(columns={"Tail_Alt_ID": "Tail_detail_name"}, inplace=True)

ibkh_Disease_Pathway["Head_ID_IS"] = 'DOID'
ibkh_Disease_Pathway["Tail_ID_IS"] = np.where(
    ibkh_Disease_Pathway["Tail"].isna(), None,
    np.where(ibkh_Disease_Pathway["Tail"].str.startswith("KEGG"), "KEGG_ID", "Reactome_ID")
)
ibkh_Disease_Pathway["Head_type"] = "Disease"
ibkh_Disease_Pathway["Tail_type"] = "Pathway"
ibkh_Disease_Pathway["Relation"]  = ibkh_Disease_Pathway["Head_type"] + "_" + ibkh_Disease_Pathway["Tail_type"]
ibkh_Disease_Pathway["KG_Source"] = "iBKH"

ibkh_Disease_Pathway = reorder(ibkh_Disease_Pathway, KG_DESIRED_COLS + ["Tail_detail_name"])
print(f"Drug–Pathway rows: {len(ibkh_Disease_Pathway):,}")

ibkh_Disease_Pathway

Drug–Pathway rows: 1,039


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Tail_detail_name
0,DOID:2747,Disease_Pathway,KEGG:map00010,Disease,Pathway,iBKH,glycogen storage disease,Glycolysis / Gluconeogenesis,DOID,KEGG_ID,Glycolysis / Gluconeogenesis
1,DOID:9869,Disease_Pathway,KEGG:map00010,Disease,Pathway,iBKH,hereditary fructose intolerance syndrome,Glycolysis / Gluconeogenesis,DOID,KEGG_ID,Glycolysis / Gluconeogenesis
2,DOID:3649,Disease_Pathway,KEGG:map00010,Disease,Pathway,iBKH,pyruvate decarboxylase deficiency,Glycolysis / Gluconeogenesis,DOID,KEGG_ID,Glycolysis / Gluconeogenesis
7,DOID:2018,Disease_Pathway,KEGG:map00010,Disease,Pathway,iBKH,hyperinsulinism,Glycolysis / Gluconeogenesis,DOID,KEGG_ID,Glycolysis / Gluconeogenesis
10,DOID:2749,Disease_Pathway,KEGG:map00010,Disease,Pathway,iBKH,glycogen storage disease Ia,Glycolysis / Gluconeogenesis,DOID,KEGG_ID,Glycolysis / Gluconeogenesis
...,...,...,...,...,...,...,...,...,...,...,...
1931,DOID:0050451,Disease_Pathway,KEGG:map05410,Disease,Pathway,iBKH,Brugada syndrome,Hypertrophic cardiomyopathy,DOID,KEGG_ID,Hypertrophic cardiomyopathy
1934,DOID:0080551,Disease_Pathway,KEGG:map05412,Disease,Pathway,iBKH,Naxos disease,Arrhythmogenic right ventricular cardiomyopathy,DOID,KEGG_ID,Arrhythmogenic right ventricular cardiomyopathy
1935,DOID:0050451,Disease_Pathway,KEGG:map05412,Disease,Pathway,iBKH,Brugada syndrome,Arrhythmogenic right ventricular cardiomyopathy,DOID,KEGG_ID,Arrhythmogenic right ventricular cardiomyopathy
1937,DOID:0050451,Disease_Pathway,KEGG:map05414,Disease,Pathway,iBKH,Brugada syndrome,Dilated cardiomyopathy,DOID,KEGG_ID,Dilated cardiomyopathy


In [272]:
ibkh_Disease_Pathway.to_csv(os.path.join(OUT_PATH, "Disease_Pathway_ibkh.csv"), index=False)
print("Saved: Drug_Pathway_ibkh.csv")
del ibkh_Disease_Pathway


Saved: Drug_Pathway_ibkh.csv


## 16. Summary Audit

In [277]:
cols_audit = ["Relation", "Head_type", "Tail_type", "KG_Source", "Head_ID_IS", "Tail_ID_IS"]
rows = []
for fp in sorted(glob(os.path.join(OUT_PATH, "*.csv"))):
    try:
        tmp = pd.read_csv(fp)
        row = {"File": os.path.basename(fp), "Triples": len(tmp)}
        for col in cols_audit:
            if col in tmp.columns:
                row[col] = set(tmp[col].dropna().unique())
        rows.append(row)
    except Exception as e:
        print(f"  Error: {fp} — {e}")

audit_df = pd.concat([pd.DataFrame([r]) for r in rows], ignore_index=True)
display(audit_df)
print(f"\nTotal triples: {audit_df['Triples'].sum():,}")


/tmp/ipykernel_1690022/4254551960.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  tmp = pd.read_csv(fp)


,File,Triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,Chemical_Chemical_ibkh.csv,2684682,{ChemicalEntity_ChemicalEntity},{ChemicalEntity},{ChemicalEntity},{iBKH},"{Drugbank, Pubchem}","{Drugbank, Pubchem}"
1,Chemical_Disease_ibkh.csv,545470,{ChemicalEntity_Disease},{ChemicalEntity},{Disease},{iBKH},{Pubchem},{DOID}
2,Chemical_Gene_ibkh.csv,1172214,{ChemicalEntity_Gene},{ChemicalEntity},{Gene},{iBKH},"{Drugbank, Pubchem}",{NCBI_ID}
3,Disease_Disease_ibkh.csv,11022,{Disease_Disease},{Disease},{Disease},{iBKH},{DOID},{DOID}
4,Disease_Gene_ibkh.csv,12323320,{Disease_Gene},{Disease},{Gene},{iBKH},"{MESH, DOID}",{NCBI_ID}
5,Disease_Pathway_ibkh.csv,1039,{Disease_Pathway},{Disease},{Pathway},{iBKH},{DOID},"{Reactome_ID, KEGG_ID}"
6,Drug_Gene_ibkh.csv,554236,{ChemicalEntity_Gene},{ChemicalEntity},{Gene},{iBKH},{Pubchem},{NCBI_ID}
7,Drug_Pathway_ibkh.csv,1670,{ChemicalEntity_Pathway},{ChemicalEntity},{Pathway},{iBKH},{Pubchem},{KEGG_ID}
8,Gene_Anatomy_ibkh.csv,8769059,{Gene_Anatomy},{Gene},{Anatomy},{iBKH},{NCBI_ID},{UBERON_ID}
9,Gene_Gene_ibkh.csv,711649,{Gene_Gene},{Gene},{Gene},{iBKH},{NCBI_ID},{NCBI_ID}



Total triples: 29,966,578
